# RAG Implementation (Retrieval-Augmented Generation)

**Author:**Emanuel Zeder

### Overview

1.  **Ingesting:** Parsing 31 Annual Reports (PDFs) from 2021-2024.
2.  **Chunking:** Splitting documents into semantic windows (approx. 1000 characters) to preserve context.
3.  **Vectorization:** Converting text into high-dimensional vectors using `sentence-transformers`.
4.  **Indexing:** Creating a searchable database using Cosine Similarity to retrieve evidence relevant to specific claims.

### 1. Ingestion & Vectorization
The following code loads the raw PDFs, extracts text and metadata (page numbers, filenames), and computes embeddings for the entire corpus.

In [ ]:
# 1. Upgrade the installer tools
!pip install --upgrade pip setuptools wheel

!pip install faiss-cpu --only-binary :all:

In [ ]:
!pip install pdfplumber sentence-transformers faiss-cpu tqdm
!pip install ipywidgets
!pip install --upgrade torch

In [ ]:
!pip install --upgrade sentence-transformers transformers torch --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!brew install xz

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install --upgrade transformers sentence-transformers

In [ ]:
import torch, transformers

In [ ]:
import os
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import os
import pdfplumber
from tqdm import tqdm  # Creates a progress bar so you know it's working

### PDF ingestion and chunking
I pull all the PDF reports from `../data/raw` and walk through each one with `pdfplumber`, page by page. For every page, I grab the text and cut it into overlapping chunks of roughly 1,000 characters with a 100-character overlap, so I don’t slice sentences in awkward places. I skip tiny bits (under ~50 chars) to avoid headers/footers. Each chunk keeps its source filename and page number, and I collect everything into a `knowledge_base` list of dictionaries like `{text, source, page}`. That list then feeds into the embedding step. ```


In [ ]:
# 1. Define Path
# We go up one level (..) then into data/raw
PDF_DIR = "../data/raw"


def load_and_chunk_pdfs(directory):
    
    #Loads all PDFs from the directory, scrapes their text,
    #and cuts them into manageable 'chunks' with metadata.
    #Handles mixed single/two-column layouts by auto-detecting a column split per page.
    
    knowledge_base = []

    # Chunking config
    chunk_size = 1000
    overlap = 100
    gap_threshold = 40  # min horizontal gap to consider it two columns
    margin = 5          # small margin when slicing columns

    def split_page_texts(page):
        #Return a list of text blocks for the page (one or two columns).
        words = page.extract_words(use_text_flow=True, keep_blank_chars=False)
        if words:
            xs = sorted(w.get("x0", 0) for w in words)
            if len(xs) >= 2:
                diffs = [xs[i + 1] - xs[i] for i in range(len(xs) - 1)]
                max_gap = max(diffs)
                if max_gap >= gap_threshold:
                    idx = diffs.index(max_gap)
                    split_x = (xs[idx] + xs[idx + 1]) / 2
                    left_box = (0 + margin, 0 + margin, split_x - margin, page.height - margin)
                    right_box = (split_x + margin, 0 + margin, page.width - margin, page.height - margin)

                    left_text = page.within_bbox(left_box).extract_text(x_tolerance=1, y_tolerance=1) or ""
                    right_text = page.within_bbox(right_box).extract_text(x_tolerance=1, y_tolerance=1) or ""

                    if len(left_text.strip()) > 20 and len(right_text.strip()) > 20:
                        return [left_text.strip(), right_text.strip()]

        fallback = page.extract_text() or ""
        return [fallback] if fallback else []

    # Get list of all PDF files
    pdf_files = [f for f in os.listdir(directory) if f.endswith('.pdf')]
    print(f"📚 Found {len(pdf_files)} reports in {directory}. Starting ingestion...")

    # Loop through every PDF (BlackRock, Amundi, etc.)
    for filename in tqdm(pdf_files):
        file_path = os.path.join(directory, filename)

        try:
            with pdfplumber.open(file_path) as pdf:
                # Loop through every page in the PDF
                for page_num, page in enumerate(pdf.pages):
                    page_texts = split_page_texts(page)

                    for text in page_texts:
                        if not text:
                            continue

                        # --- CHUNKING STRATEGY ---
                        # We split pages into chunks of ~1000 characters (approx 1 paragraph)
                        # Overlap helps ensure we don't cut a sentence in half
                        for i in range(0, len(text), chunk_size - overlap):
                            chunk_text = text[i:i + chunk_size]

                            # Only keep meaningful chunks (skip headers/footers/empty space)
                            if len(chunk_text) > 50:
                                knowledge_base.append({
                                    "text": chunk_text,
                                    "source": filename,  # Critical for citing your source later
                                    "page": page_num + 1
                                })

        except Exception as e:
            print(f"❌ Error reading {filename}: {e}")

    return knowledge_base


### Run the librarian
I look for cached outputs in `../data/processed/knowledge_base.pkl` and `embeddings.npy`. If they’re there, I just load the saved chunks and vectors so it’s instant. If not, I trigger the full build: call `load_and_chunk_pdfs` to get the chunk list, embed every chunk with `SentenceTransformer('all-MiniLM-L6-v2')`, then save both the text/metadata and the embedding matrix to disk for next time. Either way, I end up with `docs`, `db_vectors`, and a ready encoder for querying.


In [ ]:
# 2. Run the Librarian
# This might take around 30 minutes for 31 PDFs
import os
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Paths for saving your work
CACHE_DIR = "../data/processed"
DOCS_PATH = os.path.join(CACHE_DIR, "knowledge_base.pkl")
VECTORS_PATH = os.path.join(CACHE_DIR, "embeddings.npy")

# Create the directory if it doesn't exist
os.makedirs(CACHE_DIR, exist_ok=True)

def load_or_create_index():
    # 1. Check if we already have saved files
    if os.path.exists(DOCS_PATH) and os.path.exists(VECTORS_PATH):
        print("💾 Found saved index! Loading from disk...")
        
        # Load the text chunks
        with open(DOCS_PATH, "rb") as f:
            docs = pickle.load(f)
            
        # Load the vectors (math)
        db_vectors = np.load(VECTORS_PATH)
        
        print(f"✅ Loaded {len(docs)} chunks and vectors instantly.")
        return docs, db_vectors, None

    # 2. If not, we have to build it (The Long Process)
    print("⚙️ No saved index found. Building from scratch...")
    
    # Run the ingestion (Assuming load_and_chunk_pdfs is defined above)
    docs = load_and_chunk_pdfs(PDF_DIR) 
    
    # Run the embedding
    print(f"🧠 Embedding {len(docs)} chunks (this takes time)...")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    db_vectors = encoder.encode([chunk['text'] for chunk in docs])
    
    # 3. SAVE EVERYTHING so you never have to wait again
    print("💾 Saving to disk...")
    with open(DOCS_PATH, "wb") as f:
        pickle.dump(docs, f)
    np.save(VECTORS_PATH, db_vectors)
    
    print("✅ Processing complete and saved!")
    return docs, db_vectors, encoder

# --- RUN THE SMART LOADER ---
docs, db_vectors, encoder = load_or_create_index()

# If encoder wasn't loaded (because we loaded from disk), load it now for the query part
if encoder is None:
    print("Loading model for query encoding...")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')

print(f"\n✅ Ingestion Complete!")
print(f"Total Searchable Chunks: {len(docs)}")
print("Example Chunk:", docs[0])

### Reviewing Claims

In [ ]:
claims = [
    "Amundi is the leading European asset manager and among the top global players.",
    "2024 was a record financial year with higher assets under management, strong inflows, and double-digit net income growth.",
]

### Retrieval Engine: Semantic Search and Prompt Generation

This engine utilizes the vector index (`db_vectors`) to perform **semantic search**, retrieving the most relevant textual evidence from the annual reports for any given greenwashing claim. It then formats this evidence into a structured prompt for the final LLM verification step.

In [ ]:
# Assuming docs, db_vectors, and encoder are defined and loaded from Block 1.
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Pre-normalize vectors once for faster cosine similarity
_db_norm = None
if 'db_vectors' in globals():
    norms = np.linalg.norm(db_vectors, axis=1, keepdims=True) + 1e-12
    _db_norm = db_vectors / norms


def retrieve_evidence(claim, k=3, min_chars=50):
    #Semantic search over the PDF chunks with cosine similarity.
    if _db_norm is None or 'docs' not in globals():
        raise ValueError("Vectors/docs not loaded. Run the ingestion step first.")

    print(f"🔎 Searching reports for evidence regarding: '{claim}'...")

    # Encode and normalize the claim vector
    query_vec = encoder.encode([claim])
    query_vec = query_vec / (np.linalg.norm(query_vec, axis=1, keepdims=True) + 1e-12)

    # Cosine similarity via dot product on normalized vectors
    scores = np.dot(_db_norm, query_vec.squeeze())

    # Top-k indices sorted by score desc
    top_k_indices = scores.argsort()[::-1][:k]

    results = []
    for idx in top_k_indices:
        chunk = docs[idx]
        score = float(scores[idx])
        text = chunk.get('text', '')
        if len(text) < min_chars:
            continue
        results.append({
            "evidence_text": text,
            "source_doc": chunk.get('source'),
            "page_num": chunk.get('page'),
            "similarity_score": score
        })

    return results


def generate_verification_prompt(claim, retrieved_evidence, max_chars=800):
    #Build the LLM prompt with the retrieved evidence.
    context_str = ""
    for i, item in enumerate(retrieved_evidence):
        text = item['evidence_text']
        if len(text) > max_chars:
            text = text[:max_chars] + " ..."
        context_str += (
            f"\n---\n[Evidence {i+1} | Source: {item['source_doc']} (p.{item['page_num']})]:\n"
            f"{text}\n"
        )

    prompt = f"""
You are a financial auditor detecting greenwashing. Your task is to verify a sustainability CLAIM based STRICTLY on the EVIDENCE retrieved from the company's annual reports.

CLAIM TO VERIFY:
"{claim}"

RETRIEVED CONTEXT FROM ANNUAL REPORTS:
{context_str}

TASK:
Based only on the context provided, determine if the claim is:
1. SUPPORTED (The evidence confirms the claim)
2. CONTRADICTED (The evidence proves the claim is false/misleading)
3. NOT FOUND (The evidence is too vague or irrelevant to confirm or deny)

State your final result and provide a short explanation citing the specific report and page number(s) from the evidence.

RESULT:
"""
    return prompt

print("✅ Retrieval System functions defined.")


### Full RAG Verification Loop and Output

This final step simulates the production environment by using the list of key claims identified by Member B's model. It then **loops** through these claims, retrieving the most relevant evidence from the 31 reports for each one. This output proves the core value of RAG: providing **source-cited, contextual evidence** for external verification.

In [ ]:
# --- RAG PIPELINE DEMO LOOP (nicer printing) ---

BANNER = "═" * 70
SUB_DIVIDER = "─" * 70

def pretty_header(text):
    print(f"\n{BANNER}\n{text}\n{BANNER}")

# Make sure your 'claims' list is populated here for testing!
if 'claims' not in globals() or not claims:
    print("⚠️ 'claims' list not found or is empty. Creating a dummy list for demonstration.")
    claims = [
        "We successfully launched a new green bond fund targeting 5 billion EUR in 2023.",
        "Our net zero commitment requires cutting portfolio emissions by 50% by 2030."
    ]

pretty_header(f"🚀 RAG VERIFICATION LOOP • {len(claims)} claim(s)")

for idx, claim in enumerate(claims, 1):
    print(f"\n{SUB_DIVIDER}\n📌 Claim {idx}/{len(claims)}\n{SUB_DIVIDER}")
    print(f"🧾 {claim}")

    evidence = retrieve_evidence(claim, k=5)
    if not evidence:
        print("⚠️ No evidence retrieved.\n")
        continue

    print("\n🔎 Top evidence")
    for i, item in enumerate(evidence, 1):
        snippet = item['evidence_text']
        if len(snippet) > 300:
            snippet = snippet[:300] + " ..."
        print(f"{i:02d}. 📄 {item['source_doc']} (p.{item['page_num']}) • sim {item['similarity_score']:.3f}")
        print(f"    \"{snippet}\"\n")

print(f"{BANNER}\n✅ RAG Verification Loop Complete\n{BANNER}")
